# Agentic Hierarchical Chunking — اجرای مرحله‌به‌مرحله

این نوت‌بوک همان کد را به بخش‌های مستقل می‌شکند تا هر بخش را در یک سلول جدا اجرا کنید
و خروجی هر مرحله را قبل از رفتن به مرحله بعد ببینید:

1. نصب و ایمپورت‌ها + تنظیمات
2. استخراج بلاک‌ها از PDF (`extract_blocks`)
3. مدل‌های خروجی ساختاریافته Agent (`BlockLabel`, `WindowResult`) + پرامپت
4. کلاس `StructureAgent` (طبقه‌بندی parent/child/body)
5. اجرای Agent روی بلاک‌های استخراج‌شده و مشاهده خروجی
6. توابع کمکی ساخت درخت + `build_chunks`
7. اجرای کامل پایپ‌لاین و بازرسی خروجی نهایی

> نکته: بخش ۱ به ماژول‌های پروژه خودتان (`app.models.Chunks`, `app.core.config`) وابسته است.
> اگر می‌خواهید این نوت‌بوک کاملاً مستقل اجرا شود، در سلول ۱ نسخه fallback (تعریف محلی
> `ParentChunk`/`ChildChunk` و `Settings`) هم گذاشته‌ام — همان را از کامنت خارج کنید.

## ۱) نصب پیش‌نیازها و تنظیمات اولیه

اگر پکیج‌ها را ندارید، این سلول را (فقط یک‌بار) اجرا کنید.

In [1]:
# در صورت نیاز، خط زیر را از کامنت خارج و اجرا کنید
# !pip install pymupdf pydantic langchain-openai --quiet
print("پیش‌نیازها بررسی شد.")

پیش‌نیازها بررسی شد.


## ۲) ایمپورت‌ها، مدل‌های داده و تنظیم LLM

اگر ماژول‌های پروژه (`app.models.Chunks`, `app.core.config`) در دسترس نیستند،
بلوک `try/except` زیر به‌صورت خودکار نسخه‌ی محلی و ساده‌ی `ParentChunk`/`ChildChunk`
و کلید API را جایگزین می‌کند تا نوت‌بوک مستقل اجرا شود.

In [8]:
from __future__ import annotations

import statistics
from typing import Literal, Optional

import fitz  # PyMuPDF
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# --- تلاش برای استفاده از مدل‌ها و تنظیمات خود پروژه ---
try:
    from app.models.Chunks import ChildChunk, ParentChunk
    from app.core.config import Settings
    API_KEY = Settings.OPENAI_API_KEY
except ImportError:
    print("⚠️ ماژول‌های پروژه پیدا نشدند؛ از نسخه محلی/مستقل استفاده می‌شود.")

    class ChildChunk(BaseModel):
        id: str
        title: str
        content: str
        parent_id: str

    class ParentChunk(BaseModel):
        id: str
        title: str
        content: str
        children: list[ChildChunk] = Field(default_factory=list)

    # کلید API خودتان را این‌جا قرار دهید (یا از متغیر محیطی بخوانید)
    import os
    API_KEY = os.environ.get("OPENAI_API_KEY", "PASTE_YOUR_API_KEY_HERE")

BASE_URL = "https://api.gapgpt.app/v1"

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    base_url=BASE_URL,
    api_key=API_KEY,
)

print("ایمپورت‌ها و تنظیمات LLM با موفقیت انجام شد.")

ایمپورت‌ها و تنظیمات LLM با موفقیت انجام شد.


## ۳) استخراج بلاک‌ها + سیگنال‌های ساختاری از PDF

تابع `extract_blocks` متن هر بلاک PDF را به همراه سیگنال‌های کمکی
(اندازه فونت نرمال‌شده `size_z`، bold بودن، تعداد کاراکتر) استخراج می‌کند.
این سیگنال‌ها بعداً به‌عنوان evidence (نه قانون قطعی) در اختیار Agent قرار می‌گیرند.

In [9]:
def extract_blocks(path: str) -> list[dict]:
    doc = fitz.open(path)
    raw_blocks = []
    for page_num, page in enumerate(doc, start=1):
        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            text, max_size = "", 0.0
            is_bold = False
            for line in block["lines"]:
                for span in line["spans"]:
                    text += span["text"] + " "
                    max_size = max(max_size, span["size"])
                    if "bold" in span.get("font", "").lower():
                        is_bold = True
            text = text.strip()
            if text:
                raw_blocks.append({
                    "text": text,
                    "font_size": round(max_size, 1),
                    "page": page_num,
                    "bold": is_bold,
                    "bbox": block["bbox"],
                })

    if not raw_blocks:
        return []

    sizes = [b["font_size"] for b in raw_blocks]
    median_size = statistics.median(sizes)
    std_size = statistics.pstdev(sizes) or 1.0

    for b in raw_blocks:
        b["size_z"] = round((b["font_size"] - median_size) / std_size, 2)
        b["char_count"] = len(b["text"])

    return raw_blocks

print("تابع extract_blocks تعریف شد.")

تابع extract_blocks تعریف شد.


### تست مرحله ۳

مسیر فایل PDF خودتان را در `PDF_PATH` قرار دهید و این سلول را اجرا کنید
تا خروجی خام بلاک‌ها را پیش از رفتن به Agent ببینید.

In [10]:
PDF_PATH = (r"D:\rag\app\data\rag_docs.pdf")  # مسیر فایل PDF خودتان را این‌جا بگذارید

blocks = extract_blocks(PDF_PATH)
print(f"تعداد بلاک‌های استخراج‌شده: {len(blocks)}")

# نمایش چند بلاک اول برای بررسی
for b in blocks[:5]:
    print(f"[page={b['page']}] size_z={b['size_z']} bold={b['bold']} :: {b['text'][:80]}")

تعداد بلاک‌های استخراج‌شده: 228
[page=1] size_z=4.88 bold=False :: ۱ .  مقدمه و پیشینه
[page=1] size_z=0.0 bold=False :: در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هو
[page=1] size_z=0.0 bold=False :: شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ ها
[page=1] size_z=0.0 bold=False :: تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن
[page=1] size_z=0.0 bold=False :: آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.


In [11]:
blocks

[{'text': '۱ .  مقدمه و پیشینه',
  'font_size': 20.0,
  'page': 1,
  'bold': False,
  'bbox': (442.2699890136719,
   36.922061920166016,
   576.2732543945312,
   65.61934661865234),
  'size_z': 4.88,
  'char_count': 19},
 {'text': 'در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل',
  'font_size': 16.0,
  'page': 1,
  'bold': False,
  'bbox': (32.15999984741211,
   79.08988952636719,
   576.1151123046875,
   101.94461059570312),
  'size_z': 0.0,
  'char_count': 90},
 {'text': 'شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان',
  'font_size': 16.0,
  'page': 1,
  'bold': False,
  'bbox': (32.15999984741211,
   106.56993103027344,
   576.193115234375,
   129.42465209960938),
  'size_z': 0.0,
  'char_count': 94},
 {'text': 'تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان',
  'font_size': 16.0,
  'page': 1,
  'bold': False,
  'bbox': (32.15999984741211,
   134.

## ۴) مدل‌های خروجی ساختاریافته Agent + پرامپت

- `BlockLabel`: برچسب یک بلاک (`parent` / `child` / `body`) + عنوان تمیزشده.
- `WindowResult`: لیست برچسب‌های یک پنجره از بلاک‌ها (خروجی ساختاریافته LLM).
- `STRUCTURE_PROMPT`: پرامپتی که قوانین راهنما (نه قطعی) را برای تشخیص سلسله‌مراتب می‌دهد.

In [12]:
class BlockLabel(BaseModel):
    index: int = Field(..., description="اندیس بلاک در همین پنجره (local index)")
    level: Literal["parent", "child", "body"] = Field(
        ...,
        description=(
            "parent = عنوان اصلی/فصل. child = زیرعنوان/بند زیرمجموعه یک parent. "
            "body = متن معمولی/ادامه توضیحات، نه یک عنوان."
        ),
    )
    title: Optional[str] = Field(
        None,
        description="فقط برای parent/child: عنوان تمیز (بدون شماره‌گذاری ابتدایی). برای body خالی بگذار.",
    )


class WindowResult(BaseModel):
    labels: list[BlockLabel]


STRUCTURE_PROMPT = """شما یک Agent تحلیل ساختار سند هستید. وظیفه شما تشخیص این است که هر
بلاک متنی زیر، در سلسله‌مراتب سند چه نقشی دارد:

- parent : یک عنوان اصلی/فصل/بخش بزرگ سند
- child  : یک زیرعنوان یا بند که زیرمجموعه‌ی یک parent است
- body   : متن معمولی، توضیح، یا ادامه‌ی یک بند (عنوان محسوب نمی‌شود)

راهنما (نه قانون قطعی؛ گاهی سند این الگوها را ندارد و باید از روی محتوا و لحن متن قضاوت کنید):
- معمولاً parent کوتاه‌تر، پررنگ‌تر (bold) یا با فونت بزرگ‌تر از متن اطراف است.
- child معمولاً شماره‌گذاری دارد (۵.۲ / ماده ۵ / بند ۳ / الف) / a)) یا شبیه یک عنوان کوتاه است
  که با توضیح ادامه پیدا می‌کند، اما بزرگی/پررنگی آن کمتر از parent است.
- body می‌تواند بلندتر و بدون شماره‌گذاری باشد.

{context_hint}

بلاک‌های این پنجره (هرکدام با اندیس محلی خودشان و سیگنال‌های کمکی [font_z, bold]):
{items}

نکته مهم: بلاک‌هایی که با «(قبلاً برچسب‌خورده)» مشخص شده‌اند فقط برای حفظ زمینه (context) آمده‌اند،
لازم نیست دوباره برای آن‌ها پاسخ بدهید؛ فقط برای بلاک‌های بدون این برچسب پاسخ بدهید."""

print("مدل‌های ساختاریافته و پرامپت تعریف شدند.")

مدل‌های ساختاریافته و پرامپت تعریف شدند.


## ۵) کلاس `StructureAgent`

Agent با پنجره‌های هم‌پوشان (sliding window with overlap) روی بلاک‌ها حرکت می‌کند
و بین پنجره‌ها «حافظه»ی آخرین parent/child دیده‌شده را حفظ می‌کند تا برچسب‌ها در کل
سند consistent بمانند. اگر فراخوانی LLM خطا بدهد، از یک fallback ابتکاری
(بر اساس فونت/bold) استفاده می‌کند.

In [9]:
class StructureAgent:
    """
    Agent سلسله‌مراتبی که با پنجره‌های هم‌پوشان روی بلاک‌های سند حرکت می‌کند
    و با حفظ حافظه (آخرین parent/child دیده‌شده)، برچسب هر بلاک را مشخص می‌کند.
    """

    def __init__(self, window_size: int = 45, overlap: int = 8, preview_chars: int = 220):
        self.window_size = window_size
        self.overlap = overlap
        self.preview_chars = preview_chars
        self.structured_llm = llm.with_structured_output(WindowResult)

    def _build_context_hint(self, last_parent: str | None, last_child: str | None) -> str:
        if not last_parent and not last_child:
            return "این ابتدای سند است؛ هنوز هیچ parent/child‌ای تشخیص داده نشده."
        return (
            f"زمینه فعلی سند (از پنجره‌های قبلی): آخرین عنوان اصلی (parent) دیده‌شده = "
            f"«{last_parent or 'نامشخص'}», آخرین زیرعنوان (child) دیده‌شده = «{last_child or 'نامشخص'}». "
            "اگر بلاک جدیدی شروع بخش/بند تازه‌ای نباشد، همچنان زیرمجموعه همین زمینه محسوب می‌شود."
        )

    def _classify_window(
        self,
        window_blocks: list[dict],
        already_labeled: set[int],
        context_hint: str,
    ) -> dict[int, tuple[str, str | None]]:
        items_lines = []
        for local_i, b in enumerate(window_blocks):
            tag = " (قبلاً برچسب‌خورده)" if local_i in already_labeled else ""
            preview = b["text"][: self.preview_chars]
            items_lines.append(
                f"[{local_i}] font_z={b['size_z']} bold={b['bold']}{tag}: {preview}"
            )
        prompt = STRUCTURE_PROMPT.format(
            context_hint=context_hint,
            items="\n".join(items_lines),
        )

        try:
            result = self.structured_llm.invoke(prompt)
        except Exception:
            # fail-safe: اگر Agent در دسترس نبود، با سیگنال ساختاری ساده تخمین می‌زنیم
            return {
                local_i: self._heuristic_fallback(b)
                for local_i, b in enumerate(window_blocks)
                if local_i not in already_labeled
            }

        out: dict[int, tuple[str, str | None]] = {}
        for item in result.labels:
            if 0 <= item.index < len(window_blocks):
                out[item.index] = (item.level, item.title)
        return out

    @staticmethod
    def _heuristic_fallback(b: dict) -> tuple[str, str | None]:
        """فقط در صورت خطای Agent استفاده می‌شود؛ صرفاً یک تخمین خام است."""
        if b["size_z"] > 1.5 or (b["bold"] and b["char_count"] < 60):
            return "parent", b["text"][:80]
        if b["char_count"] < 60 and b["bold"]:
            return "child", b["text"][:80]
        return "body", None

    def classify_document(self, blocks: list[dict]) -> list[dict]:
        n = len(blocks)
        labeled: dict[int, tuple[str, str | None]] = {}
        last_parent_title: str | None = None
        last_child_title: str | None = None

        start = 0
        while start < n:
            end = min(start + self.window_size, n)
            window_blocks = blocks[start:end]

            # اندیس‌های محلی که در پنجره قبلی از قبل برچسب خورده‌اند (overlap)
            already_local = {
                gi - start for gi in range(start, end) if gi in labeled
            }

            context_hint = self._build_context_hint(last_parent_title, last_child_title)
            window_labels = self._classify_window(window_blocks, already_local, context_hint)

            for local_i, (level, title) in window_labels.items():
                global_i = start + local_i
                if global_i in labeled:
                    continue  # قبلاً در پنجره قبل نهایی شده، تغییرش نمی‌دهیم
                labeled[global_i] = (level, title)
                if level == "parent":
                    last_parent_title = title or blocks[global_i]["text"][:80]
                    last_child_title = None
                elif level == "child":
                    last_child_title = title or blocks[global_i]["text"][:80]

            if end >= n:
                break
            start = end - self.overlap  # پنجره بعدی با overlap شروع می‌شود

        for i, b in enumerate(blocks):
            level, title = labeled.get(i, ("body", None))
            b["level"] = level
            b["title"] = title

        return blocks

print("کلاس StructureAgent تعریف شد.")

کلاس StructureAgent تعریف شد.


### تست مرحله ۵

Agent را روی بلاک‌های استخراج‌شده در مرحله ۳ اجرا کنید و برچسب هر بلاک را ببینید
(این سلول چندین بار به LLM درخواست می‌زند، بسته به تعداد بلاک‌ها و `window_size`).

In [10]:
agent = StructureAgent(window_size=45, overlap=8)
labeled_blocks = agent.classify_document(blocks)

for b in labeled_blocks[:15]:
    print(f"[{b['level']:6}] {b['title'] or '':40} :: {b['text'][:60]}")

[parent] ۱ .  مقدمه و پیشینه                      :: ۱ .  مقدمه و پیشینه
[body  ]                                          :: در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های 
[body  ]                                          :: شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را 
[body  ]                                          :: تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل 
[body  ]                                          :: آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاص
[body  ]                                          :: برای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG 
[body  ]                                          :: Generation)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی اس
[body  ]                                          :: 1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی با
[body  ]                                          :: 2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به ع
[body  ]       

## ۶) توابع کمکی ساخت درخت + `build_chunks`

این بخش deterministic است (بدون فراخوانی LLM) و خروجی برچسب‌خورده‌ی Agent را
به درخت `ParentChunk` → `children: list[ChildChunk]` تبدیل می‌کند، با تضمین اینکه:
- هر `child` حتماً زیر یک `parent` باشد (در غیر این‌صورت یک parent موقت ساخته می‌شود).
- متن‌های `body` قبل از اولین child، به‌عنوان «مقدمه» یا در صورت نبود child واقعی،
  به‌عنوان تنها child آن parent ذخیره می‌شوند.

In [12]:
def _fallback_title(text: str) -> str:
    return text.strip()[:120]


def build_chunks_from_labeled(blocks: list[dict]) -> list[ParentChunk]:
    """همان منطق build_chunks، اما بلاک‌های از قبل برچسب‌خورده را می‌گیرد
    (برای تست مرحله‌به‌مرحله در نوت‌بوک، جدا از extract+classify)."""
    parents: list[ParentChunk] = []
    current_parent: ParentChunk | None = None
    current_child: ChildChunk | None = None
    parent_counter = 0
    child_counter = 0
    pending_body: list[str] = []
    has_real_child = False

    def flush_child():
        nonlocal current_child
        if current_parent is None or current_child is None:
            return
        current_parent.children.append(current_child)
        current_parent.content += "\n" + current_child.content
        current_child = None

    def flush_pending_as_intro():
        nonlocal child_counter
        if current_parent is None or not pending_body:
            return
        child_counter += 1
        intro = ChildChunk(
            id=f"{current_parent.id}.{child_counter}",
            title=f"مقدمه ای برای: {current_parent.title}",
            content="\n".join(pending_body),
            parent_id=current_parent.id,
        )
        current_parent.children.append(intro)
        current_parent.content += "\n" + intro.content
        pending_body.clear()

    def flush_pending_as_single_child():
        nonlocal child_counter
        if current_parent is None or not pending_body:
            return
        child_counter += 1
        only_child = ChildChunk(
            id=f"{current_parent.id}.{child_counter}",
            title=current_parent.title,
            content="\n".join(pending_body),
            parent_id=current_parent.id,
        )
        current_parent.children.append(only_child)
        current_parent.content += "\n" + only_child.content
        pending_body.clear()

    for block in blocks:
        level = block["level"]
        text = block["text"]
        title = block.get("title") or _fallback_title(text)

        if level == "parent":
            flush_child()
            if current_parent is not None:
                if not has_real_child:
                    flush_pending_as_single_child()
                parents.append(current_parent)

            parent_counter += 1
            child_counter = 0
            current_parent = ParentChunk(
                id=str(parent_counter),
                title=title,
                content="",
            )
            pending_body.clear()
            has_real_child = False

        elif level == "child":
            if current_parent is None:
                # Child بدون Parent قبلی: به‌عنوان یک Parent مستقل در نظر می‌گیریم
                # تا محتوا گم نشود (repair rule)
                parent_counter += 1
                child_counter = 0
                current_parent = ParentChunk(id=str(parent_counter), title=title, content="")
                has_real_child = False

            if not has_real_child:
                flush_pending_as_intro()
            has_real_child = True
            flush_child()
            child_counter += 1
            current_child = ChildChunk(
                id=f"{current_parent.id}.{child_counter}",
                title=title,
                content=text,
                parent_id=current_parent.id,
            )

        else:  # body
            if current_parent is None:
                # سند بدون هیچ Parent‌ای شروع شده: یک Parent موقت می‌سازیم
                parent_counter += 1
                current_parent = ParentChunk(id=str(parent_counter), title="بدون عنوان", content="")
                has_real_child = False

            if not has_real_child and current_child is None:
                pending_body.append(text)
            else:
                if current_child is not None:
                    current_child.content += "\n" + text
                else:
                    pending_body.append(text)

    flush_child()
    if current_parent is not None:
        if not has_real_child:
            flush_pending_as_single_child()
        parents.append(current_parent)

    return parents


def build_chunks(path: str, window_size: int = 45, overlap: int = 8) -> list[ParentChunk]:
    """پایپ‌لاین کامل: extract -> classify -> build tree (همان تابع اصلی)."""
    doc_blocks = extract_blocks(path)
    if not doc_blocks:
        return []
    doc_agent = StructureAgent(window_size=window_size, overlap=overlap)
    doc_blocks = doc_agent.classify_document(doc_blocks)
    return build_chunks_from_labeled(doc_blocks)

print("توابع build_chunks_from_labeled و build_chunks تعریف شدند.")

توابع build_chunks_from_labeled و build_chunks تعریف شدند.


### تست مرحله ۶: ساخت درخت از بلاک‌های همین نوت‌بوک

از `labeled_blocks` که در مرحله ۵ ساختید استفاده می‌کند (بدون فراخوانی دوباره LLM).

In [16]:
parents_result = build_chunks_from_labeled(labeled_blocks)

print(f"تعداد Parent Chunk ها: {len(parents_result)}")
for p in parents_result[:20]:
    print(f"\n### Parent [{p.id}] {p.title}")
    for c in p.children:
        print(f"   - Child [{c.id}] {c.title} :: {c.content[:60]}")

تعداد Parent Chunk ها: 8

### Parent [1] ۱ .  مقدمه و پیشینه
   - Child [1.1] ۱ .  مقدمه و پیشینه :: در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های 

### Parent [2] ۲ . اجزای اصلی یک سیستم   RAG
   - Child [2.1] مقدمه ای برای: ۲ . اجزای اصلی یک سیستم   RAG :: یک سیستم RAG   از چندین مؤلفه کلیدی تشکیل شده است که هر یک ن
   - Child [2.2] ۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion) :: ۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)
اولین گا
   - Child [2.3] ۲.۲  تقسیم بندی متن (Chunking) :: ۲.۲  تقسیم بندی متن (Chunking)
پس از بارگذاری اسناد، متن ها 
   - Child [2.4] ۲.۳    تبدیل به بردار (Embedding) :: ۲.۳    تبدیل به بردار (Embedding)
مرحله بعدی، تبدیل هر قطعه 
   - Child [2.5] ۲.۴  پایگاه داده برداری (Vector Store) :: ۲.۴  پایگاه داده برداری (Vector Store)
الگوریتم های جستجو در
   - Child [2.6] ۲.۵  بازیابی و رتبه بندی (Retrieval & Reranking) :: ۲.۵  بازیابی و رتبه بندی (Retrieval & Reranking)
هنگامی که ک
   - Child [2.7] ۲.۶    تولید پاسخ (Generation) :: ۲.۶  

In [19]:
parents_result[7].children

[ChildChunk(id='8.1', title='۸ .   بهترین رویکردها و توصیه ها', content='برای ساخت یک سیستم RAG   فارسی با کیفیت باال، بر اساس تجربه ،های عملی و مطالعات موجود\nتوصیه های زیر پیشنهاد می شود:\nتمرکز جدی بر نرمال سازی متن فارسی\nاستفاده از کتابخانه هایی مانندHazm   برای یکسان سازی کاراکترها، حذف نیم فاصله های مشکل دار\nو اصالح اشکاالت نوشتاری ضروری است.\nآزمایش مدل های مختلف Embedding\nبهتر است مدل های کوچک( تر۲۵۶    تا۵۱۲  توکن) و همچنین مدل های بزرگ( تر۵۱۲    تا۱۰۲۴\nتوکن) متناسب با نوع داده تست شوند. انتخاب مدل باید بر اساس عملکرد واقعی در دامنه کاری\nانجام شود.\nتنظیم دقیق   chunk size\n،اندازه قطعات باید بر اساس نوع محتوا تعیین شود. برای متون فنیchunkهای کوچک ًتر معموال\nعملکرد بهتری دارند.\nاستفاده از Reranking برای پرسش های پیچیده\nدر پرسش های چندوجهی یا پیچیده، استفاده از مدل های Cross-Encoder یا سرویس های\nRerank (  مانند Cohere یا Anthropic) کیفیت نتایج را به طور قابل توجهی افزایش می دهد.\nطراحی   سیستم ارزیابی مستمر\nاستفاده از مجموعه ای از سواالت و پاسخ های طالیی (Golden QA Pa

## ۷) اجرای کامل پایپ‌لاین از ابتدا (یک‌مرحله‌ای)

اگر می‌خواهید کل فرایند (استخراج + طبقه‌بندی + ساخت درخت) را با یک فراخوانی روی
یک فایل PDF جدید اجرا کنید، از تابع `build_chunks` استفاده کنید.

In [ ]:
final_chunks = build_chunks(PDF_PATH, window_size=45, overlap=8)

print(f"تعداد کل Parent Chunk ها: {len(final_chunks)}")
for p in final_chunks:
    print(f"\nParent [{p.id}] {p.title}  (children={len(p.children)})")
    for c in p.children:
        print(f"   Child [{c.id}] {c.title}")

In [14]:
import fitz  # pymupdf

doc = fitz.open(r"D:\rag\app\data\rag_docs.pdf")

for page_number, page in enumerate(doc):
    print(f"--- Page {page_number+1} ---")
    text = page.get_text("text")
    print(text)

--- Page 1 ---
۱ .
 مقدمه و پیشینه 
در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه
های اصلی نرم افزارهای هوشمند تبدیل
شده
اند. این مدل
ها قادرند متون طبیعی را درک کرده، آن
ها را تحلیل کنند و پاسخ هایی دقیق و روان
 تولید نمایند. با این حال، یکی از بزرگ
ترین چالش
های این مدل
ها آن است که دانش آن ها به زمان
آموزش محدود می
شود و دسترسی مستقیمی به داده
های جدید، اختصاصی یا سازمانی ندارند.
 
 
برای رفع این محدودیت، رویکرد «بازیابی-
 افزوده تولید» یاRAG (Retrieval-Augmented 
Generation) 
 مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
 
1. در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می
شود.
 
2. در مرحله دوم، مدل زبانی از این اطالعات بازیابی
شده به عنوان زمینه (Context) استفاده می کند
تا پاسخ نهایی را تولید نماید.
 
 
معماری RAG 
 نخستین بار توسط
لوئیس و همکارانش 
 در سال۲۰۲۰
 
 معرفی شد و از آن زمان تاکنون
تحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله
پرسش و پاسخ سازمانی، چت
بات
های پشتیبانی مشتری، تحلیل اسناد حقوقی، سیستم

In [15]:
text = page.get_text("blocks")

In [23]:
blocks = extract_blocks(r"D:\rag\app\data\rag_docs.pdf")

for b in blocks[:100]:
    print("----")
    print(b["text"])

----
۱ .  مقدمه و پیشینه
----
در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل
----
شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان
----
تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان
----
آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.
----
برای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented
----
Generation)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
----
1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.
----
2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند
----
تا پاسخ نهایی را تولید نماید.
----
معماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰    معرفی شد و از آن زمان تاکنون
----
تحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله
----
پرسش و پاسخ سازمانی، چت بات های پ

In [ ]:
"""
extract_blocks - نسخه‌ی مخصوص PDFهای تک‌ستونی و متن‌محور (فارسی/RTL)
--------------------------------------------------------------------
فرض: فایل‌ها تک‌ستونی هستند و صرفاً از متن ساده (بدون چیدمان پیچیده،
جدول یا چند ستون) تشکیل شده‌اند.

روش کار:
    به‌جای اتکا به ترتیب خام content-stream (که ممکن است در برخی
    PDFها به‌هم بریزد)، بلاک‌ها را صریحاً بر اساس مختصات:
        1) اول بر اساس y (بالا -> پایین)
        2) داخل هر ردیف هم‌سطح، بر اساس x نزولی (راست -> چپ، برای RTL)
    مرتب می‌کنیم. این ترتیب دقیقاً معادل ترتیب خواندن طبیعی متن است.
"""

import statistics
import fitz  # pymupdf


def extract_blocks2(path: str, rtl: bool = True, y_tolerance: float = 3.0) -> list[dict]:
    """
    استخراج بلاک‌های متنی از یک PDF تک‌ستونی، با ترتیب تضمین‌شده‌ی خواندن.

    Args:
        path: مسیر فایل PDF
        rtl: True برای فارسی/عربی (راست‌به‌چپ)
        y_tolerance: بلاک‌هایی با اختلاف y کمتر از این مقدار، هم‌ردیف
                     در نظر گرفته می‌شوند (بر حسب واحد PDF، معمولاً پوینت)
    """
    doc = fitz.open(path)
    all_blocks = []

    for page_num, page in enumerate(doc, start=1):
        raw_blocks = []
        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            text, max_size = "", 0.0
            is_bold = False
            for line in block["lines"]:
                for span in line["spans"]:
                    text += span["text"] + " "
                    max_size = max(max_size, span["size"])
                    if "bold" in span.get("font", "").lower():
                        is_bold = True
            text = text.strip()
            if text:
                raw_blocks.append({
                    "text": text,
                    "font_size": round(max_size, 1),
                    "page": page_num,
                    "bold": is_bold,
                    "bbox": block["bbox"],
                })

        if not raw_blocks:
            continue

        # --- مرتب‌سازی صریح بر اساس مختصات (تک‌ستونی) ---
        # مرحله 1: مرتب‌سازی اولیه بر اساس y
        raw_blocks.sort(key=lambda b: b["bbox"][1])

        # مرحله 2: گروه‌بندی بلاک‌های هم‌ردیف و مرتب‌سازی افقی آن‌ها
        rows: list[list[dict]] = []
        current_row = [raw_blocks[0]]
        for b in raw_blocks[1:]:
            if abs(b["bbox"][1] - current_row[-1]["bbox"][1]) <= y_tolerance:
                current_row.append(b)
            else:
                rows.append(current_row)
                current_row = [b]
        rows.append(current_row)

        page_ordered = []
        for row in rows:
            row_sorted = sorted(row, key=lambda b: b["bbox"][0], reverse=rtl)
            page_ordered.extend(row_sorted)

        all_blocks.extend(page_ordered)

    if not all_blocks:
        return []

    sizes = [b["font_size"] for b in all_blocks]
    median_size = statistics.median(sizes)
    std_size = statistics.pstdev(sizes) or 1.0

    for b in all_blocks:
        b["size_z"] = round((b["font_size"] - median_size) / std_size, 2)
        b["char_count"] = len(b["text"])

    return all_blocks


# if __name__ == "__main__":
#     import sys
#     path = sys.argv[1] if len(sys.argv) > 1 else r"D:\rag\app\data\rag_docs.pdf"
#     blocks = extract_blocks(path)
#     for b in blocks[:100]:
#         print("----")
#         print(b["text"])

In [40]:
blocks = extract_blocks2(r"D:\rag\app\data\rag_docs.pdf")

for b in blocks[:100]:
    print("----")
    print(b["text"])

----
۱ .  مقدمه و پیشینه
----
در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل
----
شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان
----
تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان
----
آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.
----
برای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented
----
Generation)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
----
1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.
----
2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند
----
تا پاسخ نهایی را تولید نماید.
----
معماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰    معرفی شد و از آن زمان تاکنون
----
تحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله
----
پرسش و پاسخ سازمانی، چت بات های پ

In [19]:
import fitz  # PyMuPDF


def extract_blocks(path: str, rtl: bool =True, y_tolerance: float = 3.0) -> list[dict]:
    """
    استخراج بلاک‌های متنی از یک PDF تک‌ستونی با حفظ ترتیب طبیعی خواندن.

    فرضیات:
    - سند تک‌ستونی است.
    - فقط متن ساده دارد (بدون چیدمان پیچیده، جدول یا چندستونه).

    ترتیب استخراج:
    1. از بالا به پایین (Y)
    2. در هر ردیف، از راست به چپ (برای فارسی) یا چپ به راست

    خروجی:
    [
        {
            "text": "...",
            "font_size": 16.0
        },
        ...
    ]
    """

    doc = fitz.open(path)
    blocks = []

    for page in doc:
        page_blocks = []

        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue

            text_parts = []
            max_font_size = 0.0

            for line in block["lines"]:
                for span in line["spans"]:
                    text_parts.append(span["text"])
                    max_font_size = max(max_font_size, span["size"])

            text = " ".join(text_parts).strip()

            if not text:
                continue

            page_blocks.append({
                "text": text,
                "font_size": round(max_font_size, 1),
                "bbox": block["bbox"],   # فقط برای مرتب‌سازی استفاده می‌شود
            })

        if not page_blocks:
            continue

        # مرتب‌سازی اولیه از بالا به پایین
        page_blocks.sort(key=lambda b: b["bbox"][1])

        # گروه‌بندی بلاک‌های هم‌ردیف
        rows = []
        current_row = [page_blocks[0]]

        for block in page_blocks[1:]:
            if abs(block["bbox"][1] - current_row[-1]["bbox"][1]) <= y_tolerance:
                current_row.append(block)
            else:
                rows.append(current_row)
                current_row = [block]

        rows.append(current_row)

        # مرتب‌سازی افقی هر ردیف
        for row in rows:
            row.sort(key=lambda b: b["bbox"][0], reverse=rtl)

            for block in row:
                blocks.append({
                    "text": block["text"],
                    "font_size": block["font_size"],
                })

    # اضافه کردن ایندکس نهایی
    result = []

    for idx, block in enumerate(blocks):
        result.append({
            "index": idx,
            "text": block["text"],
            "font_size": block["font_size"],
        })

    doc.close()
    return result

In [20]:
blocks = extract_blocks(r"D:\rag\app\data\rag_docs.pdf")

# for b in blocks[:100]:
#     print("----")
#     print(b["text"])

In [21]:
blocks

[{'index': 0, 'text': '۱ .  مقدمه و پیشینه', 'font_size': 20.0},
 {'index': 1,
  'text': 'در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل',
  'font_size': 16.0},
 {'index': 2,
  'text': 'شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان',
  'font_size': 16.0},
 {'index': 3,
  'text': 'تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان',
  'font_size': 16.0},
 {'index': 4,
  'text': 'آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.',
  'font_size': 16.0},
 {'index': 5,
  'text': 'برای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented',
  'font_size': 16.0},
 {'index': 6,
  'text': 'Generation)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:',
  'font_size': 16.0},
 {'index': 7,
  'text': '1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.',
  'font_size': 16.0},
 {'index': 8,
 

In [50]:
import os
from langchain_openai import ChatOpenAI
from app.core.config import Settings

API_KEY = os.environ.get("OPENAI_API_KEY", "PASTE_YOUR_API_KEY_HERE")

BASE_URL = "https://api.gapgpt.app/v1"

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    base_url=BASE_URL,
    api_key=API_KEY,
)

In [51]:
from typing import Literal
from pydantic import BaseModel, Field


class BlockLabel(BaseModel):
    index: int = Field(
        ...,
        description="ایندکس بلاک ورودی"
    )

    level: Literal["parent", "child", "body"] = Field(
        ...,
        description="نوع بلاک"
    )


class ClassificationResult(BaseModel):
    labels: list[BlockLabel]

In [52]:
structured_llm = llm.with_structured_output(ClassificationResult)

In [84]:
STRUCTURE_PROMPT = """
شما مسئول تحلیل ساختار یک سند هستید.

چند بلاک متنی از یک PDF به شما داده می‌شود.
هر بلاک یک index، متن و اندازه فونت دارد.

وظیفه شما این است که برای هر بلاک دقیقاً یکی از سه برچسب زیر را تعیین کنید.

parent:
- عنوان اصلی
- شروع یک بخش یا فصل
- معمولاً کوتاه‌تر از متن عادی است.
- ممکن است شماره‌گذاری داشته باشد.
- معمولاً فونت بزرگ‌تری نسبت به متن اطراف دارد.

child:
- زیرعنوان یک parent
- معمولاً عنوان یک زیربخش است.
الگو ها :
1.1 1.2 1.3 ...
الف) ب) و ..
نکته بسیار مهم : تا یک سطح CHILD شناسایی کن و داخل یک CHILD دیگر CHILD نخواهیم داشت

body:
- متن معمولی
- توضیح
- ادامه یک پاراگراف
- عنوان محسوب نمی‌شود.

نکات:

- همه بلاک‌ها را نسبت به یکدیگر بررسی کن.
- صرفاً بزرگ بودن فونت کافی نیست.
- صرفاً شماره‌گذاری کافی نیست.
- از محتوای متن نیز استفاده کن.
- برای همه بلاک‌ها خروجی بده.
- خروجی فقط باید مطابق Schema باشد.

بلاک‌ها:

{blocks}
"""

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(STRUCTURE_PROMPT)

In [54]:
def build_blocks_prompt(blocks: list[dict]) -> str:
    lines = []

    for block in blocks:
        lines.append(
            f"[{block['index']}] "
            f"font_size={block['font_size']} "
            f"text={block['text']}"
        )

    return "\n".join(lines)

In [75]:
window = blocks[:45]

In [85]:
builder = build_blocks_prompt(window)

In [69]:
print(builder)

[0] font_size=20.0 text=۱ .  مقدمه و پیشینه
[1] font_size=16.0 text=در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل
[2] font_size=16.0 text=شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان
[3] font_size=16.0 text=تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان
[4] font_size=16.0 text=آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.
[5] font_size=16.0 text=برای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented
[6] font_size=16.0 text=Generation)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
[7] font_size=16.0 text=1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.
[8] font_size=16.0 text=2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند
[9] font_size=16.0 text=تا پاسخ نهایی را تولید نماید.
[10] font_size=16.0 text=معماری RAG  

In [86]:
classifier = prompt | structured_llm

In [71]:
window = builder[0:45]

In [87]:
result = classifier.invoke(
    {
        "blocks": builder
    }
)

In [77]:
result

ClassificationResult(labels=[BlockLabel(index=0, level='parent'), BlockLabel(index=1, level='body'), BlockLabel(index=2, level='body'), BlockLabel(index=3, level='body'), BlockLabel(index=4, level='body'), BlockLabel(index=5, level='body'), BlockLabel(index=6, level='body'), BlockLabel(index=7, level='body'), BlockLabel(index=8, level='body'), BlockLabel(index=9, level='body'), BlockLabel(index=10, level='body'), BlockLabel(index=11, level='body'), BlockLabel(index=12, level='body'), BlockLabel(index=13, level='body'), BlockLabel(index=14, level='parent'), BlockLabel(index=15, level='body'), BlockLabel(index=16, level='body'), BlockLabel(index=17, level='child'), BlockLabel(index=18, level='body'), BlockLabel(index=19, level='body'), BlockLabel(index=20, level='body'), BlockLabel(index=21, level='body'), BlockLabel(index=22, level='body'), BlockLabel(index=23, level='child'), BlockLabel(index=24, level='body'), BlockLabel(index=25, level='body'), BlockLabel(index=26, level='body'), Blo

In [88]:
for item in result.labels:
    print(item)

index=0 level='parent'
index=1 level='body'
index=2 level='body'
index=3 level='body'
index=4 level='body'
index=5 level='body'
index=6 level='body'
index=7 level='body'
index=8 level='body'
index=9 level='body'
index=10 level='body'
index=11 level='body'
index=12 level='body'
index=13 level='body'
index=14 level='parent'
index=15 level='body'
index=16 level='body'
index=17 level='child'
index=18 level='body'
index=19 level='body'
index=20 level='body'
index=21 level='body'
index=22 level='body'
index=23 level='child'
index=24 level='body'
index=25 level='body'
index=26 level='body'
index=27 level='child'
index=28 level='body'
index=29 level='body'
index=30 level='body'
index=31 level='child'
index=32 level='body'
index=33 level='body'
index=34 level='body'
index=35 level='child'
index=36 level='body'
index=37 level='body'
index=38 level='body'
index=39 level='child'
index=40 level='body'
index=41 level='body'
index=42 level='body'
index=43 level='body'
index=44 level='body'


In [107]:
from uuid import uuid4


def build_hierarchy(blocks, result):
    # index -> level
    labels = {
        item.index: item.level
        for item in result.labels
    }

    parents = []

    current_parent = None
    current_child = None

    for block in blocks:
        level = labels.get(block["index"], "body")
        text = block["text"]

        # ---------------- Parent ----------------
        if level == "parent":

            current_parent = ParentChunk(
                id=str(uuid4()),
                title=text,
                content="",
            )

            parents.append(current_parent)
            current_child = None

# ---------------- Child ----------------
        elif level == "child":

            if current_parent is None:
                continue

            current_child = ChildChunk(
                id=str(uuid4()),
                title=text,
                content="",
                parent_id=current_parent.id,
            )

            current_parent.children.append(current_child)

            # عنوان Child هم داخل Parent.content ذخیره شود
            if current_parent.content:
                current_parent.content += "\n\n" + text
            else:
                current_parent.content = text
        # ---------------- Body ----------------
        else:

            if current_parent is None:
                continue

            # همیشه به محتوای Parent اضافه می‌شود
            if current_parent.content:
                current_parent.content += "\n" + text
            else:
                current_parent.content = text

            # اگر داخل Child هستیم، به Child هم اضافه می‌شود
            if current_child is not None:
                if current_child.content:
                    current_child.content += "\n" + text
                else:
                    current_child.content = text

# -----------------------------------------
    for parent in parents:
        if not parent.children:
            parent.children.append(
                ChildChunk(
                    id=str(uuid4()),
                    title=parent.title,
                    content=parent.content,
                    parent_id=parent.id,
                )
            )

    return parents

In [108]:
parents = build_hierarchy(window, result)

In [109]:
parents

[ParentChunk(id='6ec01998-b2ec-42a6-961a-dde87afc9998', title='۱ .  مقدمه و پیشینه', content='در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل\nشده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان\nتولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان\nآموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.\nبرای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented\nGeneration)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:\n1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.\n2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند\nتا پاسخ نهایی را تولید نماید.\nمعماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰    معرفی شد و از آن زمان تاکنون\nتحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله\nپرسش و پاسخ س

In [ ]:
# """
# این ماژول قابلیت «ارسال بلاک‌ها به صورت دسته‌ای (batch)» رو به کد قبلی اضافه می‌کند.

# فرض بر این است که این‌ها از قبل در پروژه تعریف شده‌اند:
# - llm / structured_llm  (ساخت مدل)
# - BlockLabel, ClassificationResult  (schema ها)
# - build_blocks_prompt(blocks)  (فرمت‌کننده)
# - build_hierarchy(blocks, result)  (سازنده Parent/Child)

# فقط کافیست STRUCTURE_PROMPT قبلی را با نسخه‌ی زیر (که یک placeholder برای hint دارد)
# جایگزین کنید و به‌جای فراخوانی مستقیم classifier.invoke(...) از تابع
# classify_document(...) استفاده کنید.
# """

# from langchain_core.prompts import ChatPromptTemplate


# # ----------------------------------------------------------------------
# # 1) پرامپت جدید: یک بخش hint برای پیوستگی بین دورها اضافه شده
# # ----------------------------------------------------------------------

# STRUCTURE_PROMPT = """
# شما مسئول تحلیل ساختار یک سند هستید.

# سند به‌صورت بخش‌بخش (batch) در اختیار شما قرار می‌گیرد.
# این بخش فعلی، ادامه‌ی بخش(های) قبلی همان سند است، نه یک سند مستقل.

# زمینه‌ی به‌جا‌مانده از بخش(های) قبلی (فقط برای حفظ پیوستگی ساختاری استفاده کن،
# برچسبی برای خود این خط‌ها تولید نکن چون جزو بلاک‌های ورودی نیستند):
# {hint}

# چند بلاک متنی از یک PDF به شما داده می‌شود.
# هر بلاک یک index، متن و اندازه فونت دارد.

# وظیفه شما این است که برای هر بلاک دقیقاً یکی از سه برچسب زیر را تعیین کنید.

# parent:
# - عنوان اصلی
# - شروع یک بخش یا فصل
# - معمولاً کوتاه‌تر از متن عادی است.
# - ممکن است شماره‌گذاری داشته باشد.
# - معمولاً فونت بزرگ‌تری نسبت به متن اطراف دارد.
# - مراحل parent نیستند


# child:
# - زیرعنوان یک parent
# - معمولاً عنوان یک زیربخش است.
# الگو ها :
# 1.1 1.2 1.3 ...
# الف) ب) و ..
# نکته بسیار مهم : تا یک سطح CHILD شناسایی کن و داخل یک CHILD دیگر CHILD نخواهیم داشت

# body:
# - متن معمولی
# - توضیح
# - ادامه یک پاراگراف

# نکات:

# - همه بلاک‌ها را نسبت به یکدیگر بررسی کن.
# - صرفاً بزرگ بودن فونت کافی نیست.
# - صرفاً شماره‌گذاری کافی نیست.
# - از محتوای متن نیز استفاده کن.
# - اگر اولین بلاک(های) این batch از نظر محتوا ادامه‌ی همان parent/child بخش قبل
#   به‌نظر می‌رسند (طبق زمینه‌ی بالا)، آن‌ها را body در نظر بگیر مگر اینکه شواهد
#   روشنی (فونت بزرگ‌تر/شماره‌گذاری جدید/محتوای متفاوت) نشان دهد که عنوان جدیدی
#   شروع شده است.
# - برای همه بلاک‌های این batch (و فقط همین بلاک‌ها) خروجی بده.
# - خروجی فقط باید مطابق Schema باشد.

# بلاک‌ها:

# {blocks}
# """

# prompt = ChatPromptTemplate.from_template(STRUCTURE_PROMPT)


# # ----------------------------------------------------------------------
# # 2) تقسیم بلاک‌ها به batch های هم‌پوشان (overlap)
# # ----------------------------------------------------------------------

# def make_batches(
#     blocks: list[dict],
#     batch_size: int = 45,
#     overlap: int = 10,
# ) -> list[list[dict]]:
#     """
#     بلاک‌ها را به دسته‌های batch_size تایی تقسیم می‌کند، طوری که هر دسته
#     overlap تای آخرِ دسته‌ی قبلی را هم دوباره شامل شود (برای حفظ context).

#     مثال: batch_size=45, overlap=10  =>  step = 35
#     batch1: [0:45]
#     batch2: [35:80]
#     batch3: [70:115]
#     ...
#     """
#     if overlap >= batch_size:
#         raise ValueError("overlap باید کوچکتر از batch_size باشد")

#     step = batch_size - overlap
#     n = len(blocks)
#     batches: list[list[dict]] = []

#     i = 0
#     while i < n:
#         batch = blocks[i : i + batch_size]
#         batches.append(batch)

#         if i + batch_size >= n:
#             break

#         i += step

#     return batches


# # ----------------------------------------------------------------------
# # 3) ساخت hint از آخرین parent/child کامیت‌شده تا الان
# # ----------------------------------------------------------------------

# def build_hint(blocks_by_index: dict[int, dict], labels: dict[int, str]) -> str:
#     last_parent_text = None
#     last_child_text = None

#     for idx in sorted(labels.keys()):
#         level = labels[idx]
#         text = blocks_by_index[idx]["text"]

#         if level == "parent":
#             last_parent_text = text
#             last_child_text = None  # با parent جدید، child قبلی بی‌ربط می‌شود
#         elif level == "child":
#             last_child_text = text

#     if last_parent_text is None and last_child_text is None:
#         return "این اولین بخش سند است؛ تا اینجا هیچ parent یا child‌ای شناسایی نشده."

#     lines = []
#     if last_parent_text:
#         lines.append(f"- آخرین Parent شناسایی‌شده: {last_parent_text}")
#     if last_child_text:
#         lines.append(f"- آخرین Child شناسایی‌شده (زیرِ همان Parent): {last_child_text}")

#     return "\n".join(lines)


# # ----------------------------------------------------------------------
# # 4) اجرای کل جریان روی همه‌ی batch ها
# # ----------------------------------------------------------------------

# def classify_document(
#     blocks: list[dict],
#     classifier,               # = prompt | structured_llm  (از کد شما)
#     build_blocks_prompt_fn,   # همان build_blocks_prompt قبلی شما
#     ClassificationResultCls,  # همان ClassificationResult
#     BlockLabelCls,            # همان BlockLabel
#     batch_size: int = 45,
#     overlap: int = 10,
# ):
#     """
#     کل سند را batch به batch به مدل می‌دهد و در نهایت یک ClassificationResult
#     واحد و کامل (برای تمام index ها) برمی‌گرداند که مستقیم قابل پاس‌دادن به
#     build_hierarchy است.
#     """
#     blocks_by_index = {b["index"]: b for b in blocks}
#     batches = make_batches(blocks, batch_size=batch_size, overlap=overlap)

#     final_labels: dict[int, str] = {}
#     hint_text = "این اولین بخش سند است؛ تا اینجا هیچ parent یا child‌ای شناسایی نشده."

#     for batch_num, batch in enumerate(batches, start=1):
#         batch_prompt_text = build_blocks_prompt_fn(batch)

#         result = classifier.invoke(
#             {
#                 "blocks": batch_prompt_text,
#                 "hint": hint_text,
#             }
#         )

#         # کامیت کردن نتایج این دور
#         # برای index هایی که در overlap با دور قبل مشترک‌اند، نتیجه‌ی این دور
#         # (که hint بیشتری داشته) جایگزین نتیجه‌ی قبلی می‌شود.
#         for item in result.labels:
#             final_labels[item.index] = item.level

#         # ساخت hint برای دور بعد، بر اساس آخرین وضعیت کامیت‌شده تا همین‌جا
#         hint_text = build_hint(blocks_by_index, final_labels)

#         print(f"[batch {batch_num}/{len(batches)}] "
#               f"{len(batch)} بلاک پردازش شد - hint بعدی:\n{hint_text}\n")

#     labels_list = [
#         BlockLabelCls(index=idx, level=level)
#         for idx, level in sorted(final_labels.items())
#     ]

#     return ClassificationResultCls(labels=labels_list)




In [42]:
"""
این ماژول قابلیت «ارسال بلاک‌ها به صورت دسته‌ای (batch)» رو به کد قبلی اضافه می‌کند.

نسخه‌ی اصلاح‌شده:
- بلاک‌های overlap دیگر توسط دور بعدی «دوباره لیبل‌گذاری» نمی‌شوند.
- overlap فقط به‌عنوان «بلاک‌های زمینه‌ای» (همراه با لیبلی که از قبل برایشان
  مشخص شده) به مدل نشان داده می‌شود تا ساختار را بفهمد، نه اینکه رویشان نظر بدهد.
- مدل فقط برای «بلاک‌های جدید» موظف به تولید لیبل است.
- در کد هم یک فیلتر سخت‌گیرانه هست: حتی اگر مدل اشتباهاً برای یک بلاک زمینه‌ای
  هم خروجی داد، آن نادیده گرفته می‌شود (هرگز لیبل قبلی overwrite نمی‌شود).

فرض بر این است که این‌ها از قبل در پروژه تعریف شده‌اند:
- llm / structured_llm
- BlockLabel, ClassificationResult
- build_blocks_prompt(blocks)
- build_hierarchy(blocks, result)
"""

from langchain_core.prompts import ChatPromptTemplate


# ----------------------------------------------------------------------
# 1) پرامپت جدید: بلاک‌های زمینه‌ای (context) از بلاک‌های جدید (new) جدا شده‌اند
# ----------------------------------------------------------------------

STRUCTURE_PROMPT = """
شما مسئول تحلیل ساختار یک سند هستید.

سند به‌صورت بخش‌بخش (batch) در اختیار شما قرار می‌گیرد.
این بخش فعلی، ادامه‌ی بخش(های) قبلی همان سند است، نه یک سند مستقل.

خلاصه‌ی وضعیت تا پیش از این batch (برای فهم اینکه الان زیر کدام parent/child
هستیم استفاده کن):
{hint}

بلاک‌های زمینه‌ای (Context Blocks):
این بلاک‌ها قبلاً توسط شما در دور(های) قبل لیبل‌گذاری شده‌اند و لیبلشان قطعی
و نهایی است. آنها را فقط برای فهم ساختار و پیوستگی متن بخوان.
مطلقاً برای این بلاک‌ها در خروجی چیزی تولید نکن و لیبلشان را تغییر نده.

{context_blocks}

بلاک‌های جدید (New Blocks):
فقط و فقط برای این بلاک‌ها باید دقیقاً یکی از سه برچسب زیر را تعیین کنی.
برای تمام index های این بخش، و فقط همین index ها، خروجی بده.

{new_blocks}

تعریف برچسب‌ها:

parent:
- عنوان اصلی
- شروع یک بخش یا فصل
- معمولاً کوتاه‌تر از متن عادی است.
- ممکن است شماره‌گذاری داشته باشد.
- معمولاً فونت بزرگ‌تری نسبت به متن اطراف دارد.
- مراحل parent نیستند

child:
- زیرعنوان یک parent
- معمولاً عنوان یک زیربخش است.
الگو ها :
1.1 1.2 1.3 ...
الف) ب) و ..
نکته بسیار مهم : تا یک سطح CHILD شناسایی کن و داخل یک CHILD دیگر CHILD نخواهیم داشت

body:
- متن معمولی
- توضیح
- ادامه یک پاراگراف

نکات:
- همه بلاک‌ها (زمینه‌ای + جدید) را نسبت به یکدیگر بررسی کن تا تصمیم درست بگیری،
  اما فقط برای بلاک‌های جدید خروجی بده.
- صرفاً بزرگ بودن فونت کافی نیست؛ صرفاً شماره‌گذاری هم کافی نیست.
- از محتوای متن نیز استفاده کن.
- اگر اولین بلاک(های) جدید این batch از نظر محتوا ادامه‌ی همان parent/child
  زمینه‌ای به‌نظر می‌رسند، آنها را body در نظر بگیر مگر شواهد روشنی (فونت
  بزرگ‌تر / شماره‌گذاری جدید / محتوای متفاوت) نشان دهد عنوان جدیدی شروع شده.
- خروجی فقط باید مطابق Schema باشد و دقیقاً به تعداد و index بلاک‌های جدید باشد.
"""

prompt = ChatPromptTemplate.from_template(STRUCTURE_PROMPT)


# ----------------------------------------------------------------------
# 2) تقسیم بلاک‌ها به (context, new) به‌جای یک لیست تخت با overlap
# ----------------------------------------------------------------------

def make_batches_with_context(
    blocks: list[dict],
    batch_size: int = 45,
    overlap: int = 10,
) -> list[dict]:
    """
    خروجی: لیستی از دیکشنری‌های {"context": [...], "new": [...]}

    - در batch اول: context خالی است، new شامل batch_size بلاک اول است.
    - در batch های بعدی: context شامل overlap بلاک آخرِ new دور قبل است
      (که قبلاً لیبل شده‌اند) و new شامل بلاک‌های واقعاً تازه است.

    مثال: batch_size=45, overlap=10  =>  step = 35
    batch1: new=[0:45]                context=[]
    batch2: new=[45:80]  context=[35:45]
    batch3: new=[80:115] context=[70:80]
    ...
    توجه: context و new هرگز index مشترک ندارند => امکان overwrite وجود ندارد.
    """
    if overlap >= batch_size:
        raise ValueError("overlap باید کوچکتر از batch_size باشد")

    step = batch_size - overlap
    n = len(blocks)
    batches: list[dict] = []

    start = 0
    is_first = True

    while start < n:
        if is_first:
            context = []
            new_blocks = blocks[start : start + batch_size]
            window_end = start + batch_size
        else:
            context = blocks[start : start + overlap]
            new_blocks = blocks[start + overlap : start + batch_size]
            window_end = start + batch_size

        batches.append({"context": context, "new": new_blocks})

        if window_end >= n:
            break

        start += step
        is_first = False

    return batches


# ----------------------------------------------------------------------
# 3) فرمت کردن بلاک‌های زمینه‌ای همراه با لیبل قطعی‌شان
# ----------------------------------------------------------------------

def build_context_prompt(context_blocks: list[dict], final_labels: dict[int, str]) -> str:
    if not context_blocks:
        return "(بدون بلاک زمینه‌ای - این اولین بخش سند است)"

    lines = []
    for block in context_blocks:
        label = final_labels.get(block["index"], "body")
        lines.append(
            f"[{block['index']}] "
            f"font_size={block['font_size']} "
            f"label={label} "
            f"text={block['text']}"
        )
    return "\n".join(lines)


# ----------------------------------------------------------------------
# 4) ساخت hint کلی (آخرین parent/child قطعی‌شده تا این لحظه، حتی اگر
#    خارج از پنجره‌ی overlap فعلی باشد)
# ----------------------------------------------------------------------

def build_hint(blocks_by_index: dict[int, dict], labels: dict[int, str]) -> str:
    last_parent_text = None
    last_child_text = None

    for idx in sorted(labels.keys()):
        level = labels[idx]
        text = blocks_by_index[idx]["text"]

        if level == "parent":
            last_parent_text = text
            last_child_text = None
        elif level == "child":
            last_child_text = text

    if last_parent_text is None and last_child_text is None:
        return "این اولین بخش سند است؛ تا اینجا هیچ parent یا child‌ای شناسایی نشده."

    lines = []
    if last_parent_text:
        lines.append(f"- آخرین Parent قطعی‌شده: {last_parent_text}")
    if last_child_text:
        lines.append(f"- آخرین Child قطعی‌شده (زیرِ همان Parent): {last_child_text}")

    return "\n".join(lines)


# ----------------------------------------------------------------------
# 5) اجرای کل جریان روی همه‌ی batch ها
# ----------------------------------------------------------------------

def classify_document(
    blocks: list[dict],
    classifier,               # = prompt | structured_llm
    build_blocks_prompt_fn,   # همان build_blocks_prompt قبلی شما
    ClassificationResultCls,  # همان ClassificationResult
    BlockLabelCls,            # همان BlockLabel
    batch_size: int = 45,
    overlap: int = 10,
):
    blocks_by_index = {b["index"]: b for b in blocks}
    batches = make_batches_with_context(blocks, batch_size=batch_size, overlap=overlap)

    final_labels: dict[int, str] = {}

    for batch_num, batch in enumerate(batches, start=1):
        context_blocks = batch["context"]
        new_blocks = batch["new"]
        new_indices = {b["index"] for b in new_blocks}

        hint_text = build_hint(blocks_by_index, final_labels)
        context_prompt_text = build_context_prompt(context_blocks, final_labels)
        new_prompt_text = build_blocks_prompt_fn(new_blocks)

        result = classifier.invoke(
            {
                "hint": hint_text,
                "context_blocks": context_prompt_text,
                "new_blocks": new_prompt_text,
            }
        )

        # --- فیلتر سخت‌گیرانه ---
        # فقط لیبل بلاک‌هایی که واقعاً جزو "new" این دور بودند را commit کن.
        # هر چیزی خارج از new_indices (مثلاً لیبل اشتباهیِ یک context block)
        # به‌طور کامل نادیده گرفته می‌شود و هرگز چیزی overwrite نمی‌شود.
        got_indices = set()
        for item in result.labels:
            if item.index in new_indices:
                final_labels[item.index] = item.level
                got_indices.add(item.index)

        # اگر مدل برای بعضی از بلاک‌های جدید خروجی نداد، پیش‌فرض امن body
        missing = new_indices - got_indices
        for idx in missing:
            final_labels[idx] = "body"

        print(
            f"[batch {batch_num}/{len(batches)}] "
            f"context={len(context_blocks)} new={len(new_blocks)} "
            f"missing={len(missing)}"
        )

    labels_list = [
        BlockLabelCls(index=idx, level=level)
        for idx, level in sorted(final_labels.items())
    ]

    return ClassificationResultCls(labels=labels_list)




In [1]:
import os
from langchain_openai import ChatOpenAI
# from app.core.config import Settings
from pydantic import BaseModel , Field 
from typing import Literal

API_KEY = os.environ.get("OPENAI_API_KEY", "PASTE_YOUR_API_KEY_HERE")

BASE_URL = "https://api.gapgpt.app/v1"

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    base_url=BASE_URL,
    api_key=API_KEY,
)





class BlockLabel(BaseModel):
    index: int = Field(
        ...,
        description="ایندکس بلاک ورودی"
    )

    level: Literal["parent", "child", "body"] = Field(
        ...,
        description="نوع بلاک"
    )


class ClassificationResult(BaseModel):
    labels: list[BlockLabel]
# ---------------------------------------------

structured_llm = llm.with_structured_output(ClassificationResult)

# -------------------------------------



# ------------------------------------

def build_blocks_prompt(blocks: list[dict]) -> str:
    lines = []

    for block in blocks:
        lines.append(
            f"[{block['index']}] "
            f"font_size={block['font_size']} "
            f"text={block['text']}"
        )

    return "\n".join(lines)


# -------------------------------------







import fitz  # PyMuPDF


def extract_blocks(path: str, rtl: bool =True, y_tolerance: float = 3.0) -> list[dict]:
    """
    استخراج بلاک‌های متنی از یک PDF تک‌ستونی با حفظ ترتیب طبیعی خواندن.

    فرضیات:
    - سند تک‌ستونی است.
    - فقط متن ساده دارد (بدون چیدمان پیچیده، جدول یا چندستونه).

    ترتیب استخراج:
    1. از بالا به پایین (Y)
    2. در هر ردیف، از راست به چپ (برای فارسی) یا چپ به راست

    خروجی:
    [
        {
            "text": "...",
            "font_size": 16.0
        },
        ...
    ]
    """

    doc = fitz.open(path)
    blocks = []

    for page in doc:
        page_blocks = []

        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue

            text_parts = []
            max_font_size = 0.0

            for line in block["lines"]:
                for span in line["spans"]:
                    text_parts.append(span["text"])
                    max_font_size = max(max_font_size, span["size"])

            text = " ".join(text_parts).strip()

            if not text:
                continue

            page_blocks.append({
                "text": text,
                "font_size": round(max_font_size, 1),
                "bbox": block["bbox"],   # فقط برای مرتب‌سازی استفاده می‌شود
            })

        if not page_blocks:
            continue

        # مرتب‌سازی اولیه از بالا به پایین
        page_blocks.sort(key=lambda b: b["bbox"][1])

        # گروه‌بندی بلاک‌های هم‌ردیف
        rows = []
        current_row = [page_blocks[0]]

        for block in page_blocks[1:]:
            if abs(block["bbox"][1] - current_row[-1]["bbox"][1]) <= y_tolerance:
                current_row.append(block)
            else:
                rows.append(current_row)
                current_row = [block]

        rows.append(current_row)

        # مرتب‌سازی افقی هر ردیف
        for row in rows:
            row.sort(key=lambda b: b["bbox"][0], reverse=rtl)

            for block in row:
                blocks.append({
                    "text": block["text"],
                    "font_size": block["font_size"],
                })

    # اضافه کردن ایندکس نهایی
    result = []

    for idx, block in enumerate(blocks):
        result.append({
            "index": idx,
            "text": block["text"],
            "font_size": block["font_size"],
        })

    doc.close()
    return result



# -------------------------------------------




from uuid import uuid4
from app.models.Chunks import ParentChunk , ChildChunk

def build_hierarchy(blocks, result):
    # index -> level
    labels = {
        item.index: item.level
        for item in result.labels
    }

    parents = []

    current_parent = None
    current_child = None

    for block in blocks:
        level = labels.get(block["index"], "body")
        text = block["text"]

        # ---------------- Parent ----------------
        if level == "parent":

            current_parent = ParentChunk(
                id=str(uuid4()),
                title=text,
                content="",
            )

            parents.append(current_parent)
            current_child = None

# ---------------- Child ----------------
        elif level == "child":

            if current_parent is None:
                continue

            current_child = ChildChunk(
                id=str(uuid4()),
                title=text,
                content="",
                parent_id=current_parent.id,
            )

            current_parent.children.append(current_child)

            # عنوان Child هم داخل Parent.content ذخیره شود
            if current_parent.content:
                current_parent.content += "\n\n" + text
            else:
                current_parent.content = text
        # ---------------- Body ----------------
        else:

            if current_parent is None:
                continue

            # همیشه به محتوای Parent اضافه می‌شود
            if current_parent.content:
                current_parent.content += "\n" + text
            else:
                current_parent.content = text

            # اگر داخل Child هستیم، به Child هم اضافه می‌شود
            if current_child is not None:
                if current_child.content:
                    current_child.content += "\n" + text
                else:
                    current_child.content = text

# -----------------------------------------
    for parent in parents:
        if not parent.children:
            parent.children.append(
                ChildChunk(
                    id=str(uuid4()),
                    title=parent.title,
                    content=parent.content,
                    parent_id=parent.id,
                )
            )

    return parents

# --------------------------------------------------


In [43]:
blocks = extract_blocks(r"D:\rag\app\data\rag_docs.pdf")

classifier = prompt | structured_llm  

result = classify_document(
    blocks=blocks,
    classifier=classifier,
    build_blocks_prompt_fn=build_blocks_prompt,
    ClassificationResultCls=ClassificationResult,
    BlockLabelCls=BlockLabel,
    batch_size=45,
    overlap=10,
)

parents = build_hierarchy(blocks, result)


[batch 1/7] context=0 new=45 missing=0
[batch 2/7] context=10 new=35 missing=0
[batch 3/7] context=10 new=35 missing=0
[batch 4/7] context=10 new=35 missing=0
[batch 5/7] context=10 new=35 missing=0
[batch 6/7] context=10 new=35 missing=0
[batch 7/7] context=10 new=8 missing=0


In [32]:
parents

[ParentChunk(id='401fa22d-1f3b-4b97-85be-da4d0d7d21da', title='۱ .  مقدمه و پیشینه', content='در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل\nشده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان\nتولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان\nآموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.\nبرای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented\nGeneration)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:\n1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.\n2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند\nتا پاسخ نهایی را تولید نماید.\nمعماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰    معرفی شد و از آن زمان تاکنون\nتحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله\nپرسش و پاسخ س

In [44]:
parents

[ParentChunk(id='057684fa-b589-4cec-8487-23369621f35a', title='۱ .  مقدمه و پیشینه', content='در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل\nشده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان\nتولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان\nآموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.\nبرای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented\nGeneration)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:\n1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.\n2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند\nتا پاسخ نهایی را تولید نماید.\nمعماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰    معرفی شد و از آن زمان تاکنون\nتحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله\nپرسش و پاسخ س

In [49]:
parents[1].children

[ChildChunk(id='1045774c-9a51-4c58-9024-a5874b7ddb46', title='۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)', content='اولین گام در ساخت یک سیستم RAG، آماده سازی و بارگذاری اسناد است. در این مرحله، متن خام از\nاسناد استخراج شده، پاک سازی می شود و برای مراحل بعدی آماده می گردد.\nاسناد می توانند در قالب های مختلفی مانند PDF   ، Word   ، HTML، متن ساده و حتی پایگاه های داده\nساخت یافته ارائه شوند. پس از استخراج محتوا، داده ها پردازش شده و برای استفاده در مراحل بعدی\nسیستم آماده می شوند.', parent_id='f59439f0-e3f2-47a2-9160-d880e16c1ccc'),
 ChildChunk(id='23553b97-19d9-4449-928e-b910ff8f80d6', title='۲.۲  تقسیم بندی متن (Chunking)', content='پس از بارگذاری اسناد، متن ها به قطعات کوچک تری به نامChunk تقسیم می شوند. این کار به دلیل\nمحدودیت پنجره زمینه (Context Window) مدل های زبانی ضروری است.\nاستراتژی های متداول تقسیم بندی عبارت اند از:\nالف) تقسیم بندی با اندازه ثابت (Fixed-size Chunking)\nدر این روش، متن به قطعاتی با تعداد ثابتی از کاراکترها یا توکن ها تقسیم می شود. این روش ساده و\

In [ ]:
import app.services
print("2")

In [1]:
from app.loader import extract_blocks, classify_document, build_hierarchy, prompt, structured_llm

/home/son/Desktop/rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
blocks = extract_blocks(r"/home/son/Desktop/rag/app/data/rag_docs.pdf")
classifier = prompt | structured_llm

In [3]:
result = classify_document(blocks, classifier)
parents = build_hierarchy(blocks, result)

[batch 1/7] context=0 new=45 missing=0
[batch 2/7] context=10 new=35 missing=0
[batch 3/7] context=10 new=35 missing=0
[batch 4/7] context=10 new=35 missing=0
[batch 5/7] context=10 new=35 missing=0
[batch 6/7] context=10 new=35 missing=0
[batch 7/7] context=10 new=8 missing=0


In [7]:
parents[1].children

[ChildChunk(id='99d27a75-23b4-4c57-b20e-c9ab2e010dcb', title='۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)', content='اولین گام در ساخت یک سیستم RAG، آماده سازی و بارگذاری اسناد است. در این مرحله، متن خام از\nاسناد استخراج شده، پاک سازی می شود و برای مراحل بعدی آماده می گردد.\nاسناد می توانند در قالب های مختلفی مانند PDF   ، Word   ، HTML، متن ساده و حتی پایگاه های داده\nساخت یافته ارائه شوند. پس از استخراج محتوا، داده ها پردازش شده و برای استفاده در مراحل بعدی\nسیستم آماده می شوند.', parent_id='c2c9a7a9-ea58-4a77-bdf8-eb9b50b5ed84'),
 ChildChunk(id='90e42a55-df95-4522-a949-7839d04db8c0', title='۲.۲  تقسیم بندی متن (Chunking)', content='پس از بارگذاری اسناد، متن ها به قطعات کوچک تری به نامChunk تقسیم می شوند. این کار به دلیل\nمحدودیت پنجره زمینه (Context Window) مدل های زبانی ضروری است.\nاستراتژی های متداول تقسیم بندی عبارت اند از:', parent_id='c2c9a7a9-ea58-4a77-bdf8-eb9b50b5ed84'),
 ChildChunk(id='b40b59a5-cd6f-4af2-bc1a-3c672afb212e', title='الف) تقسیم بندی با اندازه ثابت (Fi

In [5]:
from app.services.embedder import embed_and_store
b = embed_and_store(parents , 'child_test' , 'parent_test')

In [6]:
b

{'parents': 10, 'children': 25}

In [3]:
from app.loader import make_batches_with_context
a = make_batches_with_context(blocks)

In [5]:
a

[{'context': [],
  'new': [{'index': 0, 'text': '۱ .  مقدمه و پیشینه'},
   {'index': 1,
    'text': 'در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل'},
   {'index': 2,
    'text': 'شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان'},
   {'index': 3,
    'text': 'تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان'},
   {'index': 4,
    'text': 'آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.'},
   {'index': 5,
    'text': 'برای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented'},
   {'index': 6,
    'text': 'Generation)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:'},
   {'index': 7,
    'text': '1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.'},
   {'index': 8,
    'text': '2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند'},

In [6]:
parents

NameError: name 'parents' is not defined

In [18]:
blocks = extract_blocks(r"/home/son/Desktop/rag/app/data/rag_sample.pdf")
classifier = prompt | structured_llm
result = classify_document(blocks, classifier)
parents2 = build_hierarchy(blocks, result)

[batch 1/5] context=0 new=45 missing=0
[batch 2/5] context=10 new=35 missing=0
[batch 3/5] context=10 new=35 missing=0
[batch 4/5] context=10 new=35 missing=0
[batch 5/5] context=10 new=14 missing=0


In [7]:
parents2[6].children

NameError: name 'parents2' is not defined

In [17]:
blocks

[{'index': 0, 'text': '(RAG) ﺍﻓﺰﻭﺩﻩ ﺗﻮﻟﯿﺪ-ﺑﺎﺯﯾﺎﺑﯽ', 'font_size': 22.0},
 {'index': 1,
  'text': 'ﭘﯿﺎﺩﻩﺳﺎﺯﯼ ﺑﺮﺍﯼ ﺳﯿﺴﺘﻢﻫﺎﯼ ﻫﻮﺵ ﻣﺼﻨﻮﻋﯽ ﺳﺎﺯﻣﺎﻧﯽ',
  'font_size': 22.0},
 {'index': 2, 'text': 'ﺭﺍﻫﻨﻤﺎﯼ ﺟﺎﻣﻊ', 'font_size': 22.0},
 {'index': 3,
  'text': '۱۴۰۳  | ﺗﺎﺭﯾﺦ: ﺧﺮﺩﺍﺩRAG  | ﺗﻬﯿﻪﺷﺪﻩ ﺑﺮﺍﯼ ﺁﺯﻣﻮﻥ ﻭ ﺍﺭﺯﯾﺎﺑﯽ ﺳﯿﺴﺘﻢﻫﺎﯼ۱.۰ ﻧﺴﺨﻪ',
  'font_size': 9.0},
 {'index': 4, 'text': '. ﻣﻘﺪﻣﻪ ﻭ ﭘﯿﺸﯿﻨﻪ۱', 'font_size': 16.0},
 {'index': 5,
  'text': '.ﺗﺎ ﻟﺤﻈﻪﯼ ﺁﻣﻮﺯﺵ ﻣﺤﺪﻭﺩ ﻣﯽﺷﻮﺩ ﻭ ﻫﯿﭻ ﺩﺳﺘﺮﺳﯽ ﻣﺴﺘﻘﯿﻤﯽ ﺑﻪ ﺩﺍﺩﻩﻫﺎﯼ ﺟﺪﯾﺪ، ﺍﺧﺘﺼﺎﺻﯽ ﯾﺎ ﺳﺎﺯﻣﺎﻧﯽ ﻧﺪﺍﺭﻧﺪ',
  'font_size': 11.0},
 {'index': 6,
  'text': 'ﺩﻗﯿﻖ ﻭ ﺭﻭﺍﻥ ﺗﻮﻟﯿﺪ ﮐﻨﻨﺪ. ﺑﺎ ﺍﯾﻦ ﺣﺎﻝ، ﯾﮑﯽ ﺍﺯ ﺑﺰﺭﮒﺗﺮﯾﻦ ﭼﺎﻟﺶﻫﺎﯼ ﺍﯾﻦ ﻣﺪﻝﻫﺎ ﺍﯾﻦ ﺍﺳﺖ ﮐﻪ ﺍﻃﻼﻋﺎﺕ ﺁﻥﻫﺎ',
  'font_size': 11.0},
 {'index': 7,
  'text': 'ﻧﺮﻡﺍﻓﺰﺍﺭﻫﺎﯼ ﻫﻮﺷﻤﻨﺪ ﺗﺒﺪﯾﻞ ﺷﺪﻩﺍﻧﺪ. ﺍﯾﻦ ﻣﺪﻝﻫﺎ ﻗﺎﺩﺭﻧﺪ ﻣﺘﻦﻫﺎﯼ ﻃﺒﯿﻌﯽ ﺭﺍ ﺩﺭﮎ ﮐﺮﺩﻩ، ﺗﺤﻠﯿﻞ ﻧﻤﺎﯾﻨﺪ ﻭ ﭘﺎﺳﺦﻫﺎﯼ',
  'font_size': 11.0},
 {'index': 8,
  'text': '( ﺑﻪ ﯾﮑﯽ ﺍﺯ ﭘﺎﯾﻪﻫﺎﯼ ﺍﺻﻠﯽLLM) ﺩﺭ ﺩﻧﯿﺎﯼ ﺍﻣﺮﻭﺯ، ﻣﺪﻝﻫﺎﯼ ﺯﺑﺎﻧﯽ ﺑﺰﺭﮒ',
  'font_size': 11.0},
 {'index': 9,
  'text': '.( ﺍﺳﺘﻔﺎﺩﻩ ﻣﯽﮐﻨﺪ ﺗﺎ ﭘﺎﺳﺦ ﻧ

In [1]:
# dddddd